# 리포트 02 — 표적 모델: 자세 구조는 기하에서, 레벨은 측정에서

> ### ❓ 이 편이 답하는 질문
> **드론 메쉬에서 계산한 RCS 중 무엇을 믿어도 되는가?**

### 결론
1. 엔진 수치는 건강하다 — 해석 PO 기준해 대비 최대 편차 0.201 dB ⟨outputs/sbr_kr_sweep.json : summary_div16.max_abs_db_vs_po⟩(kr 21 ⟨outputs/sbr_kr_sweep.json : summary_div16.n_points⟩점 × 입사 48 ⟨outputs/sbr_kr_sweep.json : meta.n_incidence⟩방향, kr=1 까지 전부).
2. 그래서 **자세 구조**(각도에 따른 σ 의 모양)는 기하에서 나온 것으로 방어된다 — 부품별 재질 + 광선추적 가림(가림만으로 -4.24 dB ⟨outputs/report2_waveform_rcs.json : occlusion.occlusion_db⟩).
3. **레벨과 주파수 의존성은 방어되지 않는다** — 우리 밴드 기울기는 0.742 ⟨outputs/report02_derived.json : band_slope.ours_min⟩~1.699 dB/GHz ⟨outputs/report02_derived.json : band_slope.ours_max⟩ 로 측정 문헌의 3.5 ⟨outputs/report02_derived.json : band_slope.ratio_min⟩~8.1 ⟨outputs/report02_derived.json : band_slope.ratio_max⟩배 가파르다. PO 가 f² 정반사항만 남기고 모서리 회절항(PTD)을 버리기 때문이다.
4. 그래서 레벨·주파수는 **측정 앵커**에서 받는다(§4) — 밴드별 보정 -2.41 ⟨outputs/report02_derived.json : anchor.correction_min_db⟩~+2.70 dB ⟨outputs/report02_derived.json : anchor.correction_max_db⟩, 각도패턴 변화 1.9e-15 dB ⟨outputs/report02_derived.json : anchor.shape_invariance_max_abs_db⟩.
5. ⚠ 앵커는 실험실 한 곳·기체 한 대이고 최대 미통제항이 9.50 dB ⟨outputs/report02_derived.json : anchor.largest_uncontrolled_db⟩ 로 보정량보다 크다. **재보정은 검증이 아니다.**

### ✅ 주장하는 것 / ❌ 주장하지 않는 것

| ✅ 이 편이 주장하는 것 | ❌ 이 편이 주장하지 않는 것 |
|---|---|
| 자세에 따른 σ 의 **구조** — 부품별 재질 + 광선추적 가림에서 나온다 | **절대 σ 레벨** — 측정 앵커(Das, IEEE WCL 2026)에서 온다 |
| 커널이 해석 PO 기준해를 재현한다 — kr=1..100 전 구간 편차 ≤0.201 dB ⟨outputs/sbr_kr_sweep.json : summary_div16.max_abs_db_vs_po⟩ | **주파수 기울기** — PTD 회절항이 없어 계통적으로 가파르다(§4) |
| 가림·재질·부품이 σ 를 얼마나 움직이는지 — 같은 메쉬에서 켜고 끈 차이 | 널(널 깊이·자세별 최소값) — 격자·평활에 10 dB 넘게 흔들린다 |
| **모노스태틱** 자세 패턴(방위 로브 · 방위평균) | β>45° 바이스태틱 · 마이크로도플러 — 이 편의 근거로 지지되지 않는다 |
| 재보정이 각도패턴을 건드리지 않는다 — 정규화 패턴 변화 1.9e-15 dB ⟨outputs/report02_derived.json : anchor.shape_invariance_max_abs_db⟩ | 앵커 σ 가 참값에 가깝다는 것 — 재보정은 검증이 아니다(§4.3) |

### 필요한 사전지식

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| 01 §3 | 선행연구가 표적 산란 계산을 어떻게 회피했는지 — 측정 · 피팅 · stock 방치 · 해석 블레이드 |

### 재현

```bash
# ① 근거 실험 (완료 — 산출 JSON 이 저장소에 있다)
SIONNA2_GPU=2 PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/verify_sbr_kr_sweep.py
SIONNA2_GPU=2 PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/rcs_anchor.py
# ② 측정 앵커 원장 (GPU 불필요 — 위 JSON 만 읽는다)
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/sigma_anchor.py
# ③ 이 리포트 재생성 (파생 JSON + 그림 3장 + 노트북) — GPU 불필요
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/make_report02_target.py
```

| | |
|---|---|
| 출력 | `outputs/sbr_kr_sweep.json`, `outputs/rcs_anchor.json`, `outputs/sigma_anchor.json`, `outputs/report02_derived.json` |
| 소요 | kr 스윕 363 s ⟨outputs/sbr_kr_sweep.json : meta.runtime_s⟩ · 앵커 11407 s ⟨outputs/rcs_anchor.json : meta.runtime_s⟩ (GPU 1장) · 리포트 빌드 17 s ⟨outputs/report02_derived.json : _meta.runtime_s⟩ |
| 비고 | rcs_anchor 는 기체별로 쪼개 병렬 실행 후 benchmark/merge_anchor_parts.py 로 병합한다 |

---

## §1. 일곱 대의 기체

메쉬는 사진이 아니라 **제원**에서 세운다 — 공식 외형(L×W×H)·모터 대각·프롭 지름에 맞춘 뒤 부품을 **재질 그룹**으로 나눠 유지한다(`src/drones.py:43` DroneSpec, `src/drones.py:822` build_drone).
⭐ **matrice4e · mini5pro 가 실측 대상**(06편)이므로 이 둘의 충실도가 가장 중요하다.

| 기체 | 대각 [mm] | 로터 | 프롭 [mm] | 무게 [g] | 삼각형 | 외접반경 r [m] |
|---|---|---|---|---|---|---|
| DJI Mini 5 Pro ⟨outputs/report02_derived.json : airframes.mini5pro.name⟩ | 275 ⟨outputs/report02_derived.json : airframes.mini5pro.diagonal_mm⟩ | 4 ⟨outputs/report02_derived.json : airframes.mini5pro.n_rotors⟩ | 152.4 ⟨outputs/report02_derived.json : airframes.mini5pro.prop_dia_mm⟩ | 250 ⟨outputs/report02_derived.json : airframes.mini5pro.weight_g⟩ | 29044 ⟨outputs/report02_derived.json : airframes.mini5pro.n_tris⟩ | 0.217 ⟨outputs/report02_derived.json : airframes.mini5pro.r_encl_m⟩ |
| DJI Phantom 4 ⟨outputs/report02_derived.json : airframes.phantom4.name⟩ | 350 ⟨outputs/report02_derived.json : airframes.phantom4.diagonal_mm⟩ | 4 ⟨outputs/report02_derived.json : airframes.phantom4.n_rotors⟩ | 240.0 ⟨outputs/report02_derived.json : airframes.phantom4.prop_dia_mm⟩ | 1380 ⟨outputs/report02_derived.json : airframes.phantom4.weight_g⟩ | 28402 ⟨outputs/report02_derived.json : airframes.phantom4.n_tris⟩ | 0.311 ⟨outputs/report02_derived.json : airframes.phantom4.r_encl_m⟩ |
| DJI Mavic 4 Pro ⟨outputs/report02_derived.json : airframes.mavic4pro.name⟩ | 441 ⟨outputs/report02_derived.json : airframes.mavic4pro.diagonal_mm⟩ | 4 ⟨outputs/report02_derived.json : airframes.mavic4pro.n_rotors⟩ | 267.0 ⟨outputs/report02_derived.json : airframes.mavic4pro.prop_dia_mm⟩ | 1063 ⟨outputs/report02_derived.json : airframes.mavic4pro.weight_g⟩ | 29932 ⟨outputs/report02_derived.json : airframes.mavic4pro.n_tris⟩ | 0.359 ⟨outputs/report02_derived.json : airframes.mavic4pro.r_encl_m⟩ |
| DJI Matrice 4E ⟨outputs/report02_derived.json : airframes.matrice4e.name⟩ | 439 ⟨outputs/report02_derived.json : airframes.matrice4e.diagonal_mm⟩ | 4 ⟨outputs/report02_derived.json : airframes.matrice4e.n_rotors⟩ | 274.0 ⟨outputs/report02_derived.json : airframes.matrice4e.prop_dia_mm⟩ | 1219 ⟨outputs/report02_derived.json : airframes.matrice4e.weight_g⟩ | 30714 ⟨outputs/report02_derived.json : airframes.matrice4e.n_tris⟩ | 0.358 ⟨outputs/report02_derived.json : airframes.matrice4e.r_encl_m⟩ |
| Yuneec Typhoon H (H480) ⟨outputs/report02_derived.json : airframes.typhoonh480.name⟩ | 480 ⟨outputs/report02_derived.json : airframes.typhoonh480.diagonal_mm⟩ | 6 ⟨outputs/report02_derived.json : airframes.typhoonh480.n_rotors⟩ | 230.2 ⟨outputs/report02_derived.json : airframes.typhoonh480.prop_dia_mm⟩ | 1950 ⟨outputs/report02_derived.json : airframes.typhoonh480.weight_g⟩ | 34732 ⟨outputs/report02_derived.json : airframes.typhoonh480.n_tris⟩ | 0.389 ⟨outputs/report02_derived.json : airframes.typhoonh480.r_encl_m⟩ |
| Holybro X500 V2 ⟨outputs/report02_derived.json : airframes.x500v2.name⟩ | 500 ⟨outputs/report02_derived.json : airframes.x500v2.diagonal_mm⟩ | 4 ⟨outputs/report02_derived.json : airframes.x500v2.n_rotors⟩ | 254.0 ⟨outputs/report02_derived.json : airframes.x500v2.prop_dia_mm⟩ | 1650 ⟨outputs/report02_derived.json : airframes.x500v2.weight_g⟩ | 18816 ⟨outputs/report02_derived.json : airframes.x500v2.n_tris⟩ | 0.390 ⟨outputs/report02_derived.json : airframes.x500v2.r_encl_m⟩ |
| DJI S1000+ ⟨outputs/report02_derived.json : airframes.s1000plus.name⟩ | 1045 ⟨outputs/report02_derived.json : airframes.s1000plus.diagonal_mm⟩ | 8 ⟨outputs/report02_derived.json : airframes.s1000plus.n_rotors⟩ | 381.0 ⟨outputs/report02_derived.json : airframes.s1000plus.prop_dia_mm⟩ | 9500 ⟨outputs/report02_derived.json : airframes.s1000plus.weight_g⟩ | 35370 ⟨outputs/report02_derived.json : airframes.s1000plus.n_tris⟩ | 0.730 ⟨outputs/report02_derived.json : airframes.s1000plus.r_encl_m⟩ |

![airframes](outputs/figures/report2_gallery.png)

**그림 1.** SBR 이 실제로 적분하는 메쉬는 어떻게 생겼나?

*(Sionna 자체 렌더러가 같은 메쉬를 그린 것이다 — 삽화가 아니다. 색 = 재질 그룹. 렌더는 5종만 있다.)*

### §1.1 전기적 크기 — 실측 대상이 가장 불리한 자리에 있다

PO 근사는 표적이 파장보다 충분히 커야 한다. 기체 7 × 밴드 3 = 21 ⟨outputs/report02_derived.json : electrical.n_airframe_band⟩개 조합 중 전기적으로 가장 작은 것은 **DJI Mini 5 Pro ⟨outputs/report02_derived.json : electrical.kr_min_name⟩ @ LTE ⟨outputs/report02_derived.json : electrical.kr_min_band⟩** 로 kr = 8.38 ⟨outputs/report02_derived.json : electrical.kr_min⟩, 가장 큰 것은 DJI S1000+ ⟨outputs/report02_derived.json : electrical.kr_max_name⟩ @ WiFi ⟨outputs/report02_derived.json : electrical.kr_max_band⟩ 의 kr = 79.7 ⟨outputs/report02_derived.json : electrical.kr_max⟩ 다.

PO 모델 간극이 1 dB 아래로 내려가는 지점은 kr = 9.06 ⟨outputs/report02_derived.json : po_floor.kr_below_1p0_db⟩(§3) 이고, 그보다 작은 조합이 1 ⟨outputs/report02_derived.json : electrical.n_below_po_1db⟩개 있다.
**실측 대상 하나가 PO 가 가장 불편한 구석에 앉아 있다.** 그 대가의 크기는 §3 이 잰다.

![electrical size](outputs/figures/report02_electrical_size.png)

**그림 2.** 각 기체·밴드는 PO 근사가 얼마나 편안한 영역에 놓이는가?

### §1.2 부품별 재질 — 두 엔진이 같은 표를 읽는다

Sionna RT 와 우리 SBR 적분기는 **같은 재질 표**를 읽는다(`src/materials.py:53` MATERIALS, `src/drones.py:562` DRONE_GROUP_MAT). ITU-R P.2040 이 정의하는 것은 그대로 쓰고, 없는 것(플라스틱·카본)만 문헌값으로 채운다.

| 재질 | 출처 | 쓰이는 부품 | \|Γ\| 벌크 | \|Γ\| PO 실효 |
|---|---|---|---|---|
| metal | ITU | 모터 · 배터리팩 | 1.000 ⟨outputs/report1.json : chamber.materials.metal.gamma_bulk⟩ | 1.00 ⟨outputs/report1.json : chamber.materials.metal.gamma_po⟩ |
| camera_assembly | ITU | 짐벌 카메라(금속 하우징 + 렌즈) | 1.000 ⟨outputs/report1.json : chamber.materials.camera_assembly.gamma_bulk⟩ | 0.85 ⟨outputs/report1.json : chamber.materials.camera_assembly.gamma_po⟩ |
| pcb | ITU | ESC · 메인보드(FR-4 + 구리면) | 1.000 ⟨outputs/report1.json : chamber.materials.pcb.gamma_bulk⟩ | 0.80 ⟨outputs/report1.json : chamber.materials.pcb.gamma_po⟩ |
| carbon | 문헌 | 암 · 데크 · 카본 착륙장치 | 0.989 ⟨outputs/report1.json : chamber.materials.carbon.gamma_bulk⟩ | 0.90 ⟨outputs/report1.json : chamber.materials.carbon.gamma_po⟩ |
| plastic | 문헌 | 동체 셸 · 캐노피 · 착륙장치 | 0.244 ⟨outputs/report1.json : chamber.materials.plastic.gamma_bulk⟩ | 0.28 ⟨outputs/report1.json : chamber.materials.plastic.gamma_po⟩ |
| prop_plastic | 문헌 | 프로펠러(셸보다 얇음) | 0.244 ⟨outputs/report1.json : chamber.materials.prop_plastic.gamma_bulk⟩ | 0.25 ⟨outputs/report1.json : chamber.materials.prop_plastic.gamma_po⟩ |

### §1.3 이 메쉬가 실물과 얼마나 다른가

제원에서 세운 우리 메쉬를 **실물 유래 메쉬** 넷과 같은 커널·같은 자세로 맞댄다(`benchmark/compare_community.py`, `src/compare_phantom_scan.py`).

| 대조 원본 | Δ 투영면적 [dB] | Δ 방위평균 σ [dB] | 자세별 RMS [dB] |
|---|---|---|---|
| Phantom 4 real scan (0.4mm) ⟨outputs/phantom4_scan_compare.json : name_real⟩ | +0.14 ⟨outputs/phantom4_scan_compare.json : d_area_db⟩ | -1.79 ⟨outputs/phantom4_scan_compare.json : d_sigma_db⟩ | 6.7 ⟨outputs/phantom4_scan_compare.json : d_sigma_rms_db⟩ |
| Typhoon H480 (real CAD, Apache-2.0) ⟨outputs/real_cad_compare.json : typhoon.name_real⟩ | +1.83 ⟨outputs/real_cad_compare.json : typhoon.d_area_db⟩ | +0.74 ⟨outputs/real_cad_compare.json : typhoon.d_sigma_db⟩ | 7.6 ⟨outputs/real_cad_compare.json : typhoon.d_sigma_rms_db⟩ |
| DJI Matrice 100 (2015 quad) (community mesh) ⟨outputs/community_compare.json : m100.name_real⟩ | -3.88 ⟨outputs/community_compare.json : m100.d_area_db⟩ | +0.49 ⟨outputs/community_compare.json : m100.d_sigma_db⟩ | 7.0 ⟨outputs/community_compare.json : m100.d_sigma_rms_db⟩ |
| DJI Matrice 600 Pro (2016 hexa) (community mesh) ⟨outputs/community_compare.json : m600.name_real⟩ | -3.17 ⟨outputs/community_compare.json : m600.d_area_db⟩ | +1.67 ⟨outputs/community_compare.json : m600.d_sigma_db⟩ | 10.2 ⟨outputs/community_compare.json : m600.d_sigma_rms_db⟩ |

**읽는 법**: 방위평균은 ~1 dB 안에서 맞고 **자세별 RMS 는 맞지 않는다** — 널 위치가 메쉬 세부에 민감해서다.
그래서 이 편은 로브와 방위평균만 인용한다(§5).

## §2. 스톡 Sionna 로는 안 되는 이유, 그리고 우리 엔진

Sionna 의 `PathSolver` 는 **전파(propagation)** 도구다. 표면을 국소 무한 거울로 보고(기하광학) 경로별 지연·도플러·복소이득을 준다.

**표면적분 단계가 없다** — 그래서 표적의 σ 가 창발하지 않는다. 광선을 더 쏘아도 해결되지 않는다.

| 실험 | 결과 | 출처 |
|---|---|---|
| 금속 평판 변 0.2 ⟨outputs/report3_rt.json : D_plate.rows[0].side_m⟩→4.0 m ⟨outputs/report3_rt.json : D_plate.rows[-1].side_m⟩ (σ 가 52.0 dB ⟨outputs/report3_rt.json : D_plate.sigma_span_db⟩ 변함) | RT 진폭 4.94e-06 dB ⟨outputs/report3_rt.json : D_plate.rt_span_db⟩ 변함 (= 불변) | `benchmark/rt_experiments.py` D |
| PEC 구에 광선 1M~400M 발 | 표적 경로 0 ⟨outputs/report3_rt.json : E_sphere.rows[0].n_paths⟩개 (곡면 정반사점을 거울상법이 못 찾는다) | 동 E |
| 드론 확산 에코, 광선 25M→400M | 코히런트 합이 +14.3 dB ⟨outputs/report02_derived.json : stock_rt.coh_climb_db⟩ 더 커진다 (수렴하지 않는다) | 동 A |
| ITU `metal` 의 산란계수 S | 0.0 ⟨outputs/report3_rt.json : C_metal.itu_metal_S⟩ → 모터·배터리·PCB 는 확산 기여 0 | 동 C |

σ 의 93% ⟨outputs/report3_rt.json : C_metal.metal_share_pct⟩ 가 바로 그 금속 부품에 얹혀 있다. **σ 는 적분에서 나온다.**

![stock RT](outputs/figures/report3_f6_no_sigma.png)

**그림 3.** 광선을 더 쏘면 스톡 Sionna RT 가 드론의 σ 를 내놓는가?

### §2.1 우리가 하는 것 — SBR (광선 + PO 면적분)

상용 솔버(FEKO/CST SBR+)가 고주파 RCS 를 내는 표준 방법 그대로다: **① 광선으로 실제 조명면을 찾고 ② 그 위에서 PO 표면적분**(`src/rcs_sbr.py:184`).

| 단계 | 무엇을 | 누가 |
|---|---|---|
| 첫 충돌 탐색 · 가림 | 어느 면이 실제로 보이는가 | 🟢 Sionna 의 Mitsuba/OptiX 광선엔진 |
| 재질 \|Γ\| | 부품별 반사계수 | 🟢 Sionna 재질표 (`src/materials.py:53`) |
| PO 면적분 → σ | E = Σ \|Γᵢ\| e^{j2k pᵢ·û} d², σ = 4π\|E\|²/λ² | 🔵 우리 (`src/rcs_sbr.py:184`) |
| 셸 투과 | 얇은 유전체 셸 뒤 금속(배터리·PCB) 코히런트 합 | 🔵 우리 (동 `penetrate=True`) |

게재된 선행 중 이 단계를 갖춘 것은 없다 — Proc. IEEE 2026 의 Clutter-Aware ISAC 조차 메쉬를 Sionna 에 넣고 **stock Fresnel 로 둔다**(01편 §3).

![occlusion](outputs/figures/report2_occlusion.png)

**그림 4.** 광선을 PO 앞에 두면 σ 가 얼마나 달라지는가?

한 방위에서 순수 PO 가 '조명됐다'고 센 면의 38% ⟨outputs/report2_waveform_rcs.json : occlusion.hidden_frac⟩ 는 실제로 다른 것 뒤에 있다. 방위 72 ⟨outputs/report2_waveform_rcs.json : occlusion.n_az⟩점 평균으로 -4.24 dB ⟨outputs/report2_waveform_rcs.json : occlusion.occlusion_db⟩, 오목부 다중반사는 +0.12 dB ⟨outputs/report2_waveform_rcs.json : occlusion.multibounce_db⟩ 뿐이다.

가림의 크기는 기체 형상이 정한다 — 방위 36 ⟨outputs/report6_sbr.json : n_az⟩점 격자에서 -2.69 ⟨outputs/report6_sbr.json : compare.mavic4pro.occl_el15⟩(Mavic 4 Pro) ~ +0.39 dB ⟨outputs/report6_sbr.json : compare.s1000plus.occl_el15⟩(S1000+, 열린 프레임이라 가릴 것이 없다) 다. ⚠ 두 줄은 **자세격자가 다르므로** Mavic 값도 서로 다르다 — 같은 격자 안에서만 비교할 것.

## §3. 이 엔진이 무엇만큼 값어치가 있나 — 절대 섞지 않는 두 가지

```
(커널 − Mie)  =  (커널 − 해석 PO)   +   (해석 PO − Mie)
                  ↑ (a) 우리 수치오차       ↑ (b) PO 근사를 쓴 대가
```
**(a) 는 우리 것**이고 격자를 조이면 줄어든다. **(b) 는 물리모델 고유**라 못 줄인다. 기준을 하나로 두면 (b) 가 통째로 우리 오차로 잘못 기입된다.

⭐ **Mie 와 해석 PO 는 둘 다 기준해(reference solution)이지 우리 출력이 아니다** — 닫힌형 해다(`benchmark/mie_pec_sphere.py:98` Mie, `:127` 해석 PO). 저장소 방침상 **순수 PO 를 엔진으로 쓴 결과는 리포트에 없다.**

![kr sweep](outputs/figures/report02_kr_sweep.png)

**그림 5.** 우리 수치오차와 PO 근사 자체의 간극은 각각 kr 에 따라 얼마인가?

### §3.1 두 눈금, 따로 읽는 법

|  | (a) 우리 수치오차 · 과녁 = 해석 PO | (b) PO 모델의 간극 · 과녁 = 정확 Mie |
|---|---|---|
| 최대 편차 (kr=1..100) | 0.201 dB ⟨outputs/sbr_kr_sweep.json : summary_div16.max_abs_db_vs_po⟩ | 6.73 dB ⟨outputs/sbr_kr_sweep.json : summary_div16.max_abs_db_vs_mie⟩ (kr=1) |
| kr≥30 산포 | 0.885% ⟨outputs/sbr_kr_sweep.json : summary_div16.std_sbr_over_po_pct_kr_ge30⟩ | 1.834% ⟨outputs/sbr_kr_sweep.json : summary_div16.std_sbr_over_mie_pct_kr_ge30⟩ |
| 1 dB 아래로 내려가는 kr | 해당 없음 (전 구간 0.201 dB ⟨outputs/sbr_kr_sweep.json : summary_div16.max_abs_db_vs_po⟩ 이내) | 9.06 ⟨outputs/report02_derived.json : po_floor.kr_below_1p0_db⟩ |
| 0.5 / 0.2 dB 아래로 | 해당 없음 | 15.16 ⟨outputs/report02_derived.json : po_floor.kr_below_0p5_db⟩ / 30.87 ⟨outputs/report02_derived.json : po_floor.kr_below_0p2_db⟩ |

**Sagitta(preprint)가 말하는 kr≥30 은 우리 바닥이 아니라 그들 커널이 Mie 로 수렴하는 지점**이다. 우리 커널은 kr=1 ⟨outputs/sbr_kr_sweep.json : summary_div16.kr_min⟩ 에서도 (a) 열의 편차 안에 있다.

### §3.2 ⚠ (b)는 **하한이지 상한이 아니다**

위 kr 문턱은 **매끄럽고 볼록한 구**에서 잰 값이다. 우리 표적은 그렇지 않다 — 프롭 날개·암 모서리·착륙장치처럼 **얇고 모서리가 많다**.

PO 는 모서리 회절을 담지 못하므로 실제 드론에서의 모델 간극은 이 구 값보다 **크다**. 그 크기는 우리가 모른다.

여기에 격자 불확실성이 더 붙는다 — λ/16 격자에서 서브셀 위상 산포가 1.78 dB ⟨outputs/report2_waveform_rcs.json : sbr_validation.dither[2].spread⟩(λ/24 에서 0.34 dB ⟨outputs/report2_waveform_rcs.json : sbr_validation.dither[3].spread⟩). **절대 레벨에 붙는 값이고 자세 간 상대 패턴에는 훨씬 덜 붙는다.**

## §4. ⭐ 앵커 — 왜 레벨과 주파수는 측정에서 받는가

밴드 3점으로 잰 우리 σ(f) 기울기는 0.742 ⟨outputs/report02_derived.json : band_slope.ours_min⟩(x500v2 ⟨outputs/report02_derived.json : band_slope.ours_min_drone⟩) ~ 1.699 dB/GHz ⟨outputs/report02_derived.json : band_slope.ours_max⟩(phantom4 ⟨outputs/report02_derived.json : band_slope.ours_max_drone⟩) 인데, 측정 문헌은 0.210 ⟨outputs/rcs_anchor.json : literature.mu_eps.multiband_phantom3.mu_a⟩(Das, IEEE WCL 2026) 와 0.315 dB/GHz ⟨outputs/rcs_anchor.json : literature.mu_eps.mono3d_theta90.mu_a⟩(Yuan, EuCAP 2025) 다 — 우리가 3.5 ⟨outputs/report02_derived.json : band_slope.ratio_min⟩~8.1 ⟨outputs/report02_derived.json : band_slope.ratio_max⟩배 가파르다.

**원인**: PO 는 f² 로 커지는 정반사항만 남긴다. 실제 표적에는 **주파수에 거의 무관한 모서리 회절항**이 함께 있고 우리는 그 항(PTD)을 구현하지 않았다. 부호가 정해진 계통오차이므로 **측정으로 교정할 수 있다** — 그것이 앵커다.

재보정은 `src/sigma_anchor.py` 가 하고 원장은 `outputs/sigma_anchor.json` 이다. 모드는 `slope_only ⟨outputs/report02_derived.json : anchor.mode⟩` — 크기법칙(L²/L⁴)을 쓰지 않는 쪽을 택했다(§4.3).

![band slope](outputs/figures/report02_band_slope.png)

**그림 6.** 측정 앵커는 각 기체의 밴드 기울기를 어디로 옮기는가?

### §4.1 분해 — 어느 축이 어디서 오나

게재된 표준트랙 논문이 쓰는 분해를 그대로 따른다 — σ = **A(f)·B₁(φ,θ)·B₂** (Zhang, IEEE JSAC 44:702, 2026: 측정 적합 모델).

| 인자 | 무엇 | 어디서 오나 | 이 편의 근거 |
|---|---|---|---|
| A(f) | 레벨 + 주파수 의존성 | **측정 앵커**(외부) | Das μ = 0.21 ⟨outputs/rcs_anchor.json : literature.mu_eps.multiband_phantom3.mu_a⟩·f -19.19 dBsm ⟨outputs/rcs_anchor.json : literature.mu_eps.multiband_phantom3.mu_b⟩ |
| B₁(φ,θ) | 자세에 따른 모양 | **기하**(SBR+PO) | 재보정이 바꾼 정규화 패턴 1.9e-15 dB ⟨outputs/report02_derived.json : anchor.shape_invariance_max_abs_db⟩ |
| B₂ | 산포(자세 요동의 분포) | 기하 — 단 분포족만 | 적합 RMSE 2.41 dB ⟨outputs/rcs_anchor.json : literature.fit_rmse_db.AAV⟩ 가 문헌 기준선 |

밴드별 보정량은 -2.41 ⟨outputs/report02_derived.json : anchor.correction_min_db⟩ ~ +2.70 dB ⟨outputs/report02_derived.json : anchor.correction_max_db⟩ 이고, 그동안 **모양은 소수점 15자리까지 그대로**다. 눈금만 측정에서 온다.

### §4.2 비교가능성 원장 — 보정된 숫자가 기체마다 같은 뜻이 아니다

앵커 기체는 DJI Phantom 3 ⟨outputs/report02_derived.json : anchor.anchor_platform⟩ 한 대다. 같은 급이면 'direct', 크기법칙으로 옮겼으면 'scaled', 위상 자체가 다르면 'not_comparable' 이다 — 1 ⟨outputs/report02_derived.json : anchor.n_direct⟩ / 5 ⟨outputs/report02_derived.json : anchor.n_scaled⟩ / 1 ⟨outputs/report02_derived.json : anchor.n_not_comparable⟩ 대씩이다.

| 기체 | 대각 D [m] | D/D_ref | 로터 | 판정 | L²↔L⁴ 산포 [dB] |
|---|---|---|---|---|---|
| DJI Mini 5 Pro | 0.275 | 0.79 | 4 | scaled | 2.09 |
| DJI Phantom 4 | 0.350 | 1.00 | 4 | direct | 0.00 |
| DJI Mavic 4 Pro | 0.441 | 1.26 | 4 | scaled | 2.01 |
| DJI Matrice 4E | 0.439 | 1.25 | 4 | scaled | 1.96 |
| Yuneec Typhoon H (H480) | 0.480 | 1.37 | 6 | scaled | 2.74 |
| Holybro X500 V2 | 0.500 | 1.43 | 4 | scaled | 3.10 |
| DJI S1000+ | 1.045 | 2.99 | 8 | not_comparable | 9.50 |

출처 ⟨outputs/sigma_anchor.json : drones⟩

### §4.3 ⚠ 앵커가 통제하지 못한 것 — 반드시 함께 읽을 것

| 미통제 항목 | 상태 | 크기 [dB] |
|---|---|---|
| polarisation | UNRESOLVED | 미상 |
| statistic convention (Das mu) | RESOLVED_EMPIRICALLY | +0.93 |
| size transfer law | UNRESOLVED | +9.50 |
| single platform / single lab | UNRESOLVED | 미상 |
| elevation matching | PARTIAL | +0.73 |
| near-field vs far-field, environment | OK | +0.00 |

출처 ⟨outputs/sigma_anchor.json : uncontrolled⟩

가장 큰 미통제항 **size transfer law ⟨outputs/report02_derived.json : anchor.largest_uncontrolled_term⟩** 이 9.50 dB ⟨outputs/report02_derived.json : anchor.largest_uncontrolled_db⟩ 로, 실제 적용한 보정 최대치 2.70 dB ⟨outputs/report02_derived.json : anchor.correction_abs_max_db⟩ 보다 크다. 밴드 간 격차를 **빡빡한 숫자로 읽으면 안 된다**(05편).

⭐ **측정에 맞춰 재척도하는 것은 그 측정으로 검증한 것이 아니다.** 자유도를 소모했을 뿐이고, 검증은 06편의 독립 측정이 한다. 커널 자체는 그대로다 — 원장만 있다.

## §5. 살아남는 결과 — 자세 · 부품 · 재질

위 경계를 지키면 세 가지가 남는다. 전부 **같은 메쉬 위에서 무엇을 켜고 끈 차이**라 절대 레벨의 불확실성이 상당 부분 상쇄된다.

| 살아남는 것 | 왜 | 인용해도 되는 형태 |
|---|---|---|
| 방위 로브 위치·상대 높이 | 기하가 결정한다 | 로브 · 방위평균 |
| 부품별 기여 | 같은 메쉬에서 그룹만 뺀 차이 | Δ dB |
| 재질 가중의 대가 | \|Γ\| 만 바꾼 차이 | Δ dB |
| **널 깊이** | 격자·평활·밴드평균에 10 dB 넘게 흔들린다 | ❌ 인용 금지 |

![aspect](outputs/figures/report2_rcs_polar.png)

**그림 7.** 기체와 밴드에 따라 자세 패턴의 모양이 어떻게 달라지는가?

*(로브는 안정적이고 로브 사이 널은 그렇지 않다 — 그림 안 캡션이 그 경계를 적고 있다.)*

### §5.1 부품을 하나씩 빼면

DJI Mavic 4 Pro ⟨outputs/report2_waveform_rcs.json : materials.name⟩ · 3.5 GHz ⟨outputs/report02_derived.json : bands_ghz.5G⟩ · el=15° ⟨outputs/report2_waveform_rcs.json : materials.el⟩, 같은 커널·같은 자세격자에서 그룹만 지운다(`src/viz_report2.py:1075`). **드론은 금속 덩어리가 아니다.**

| 무엇을 남겼나 | 면 | 방위평균 σ [dBsm] | Δ [dB] |
|---|---|---|---|
| Full drone (shell opaque - our SBR) | 28612 | -18.37 | +0.00 |
| - shell (RF sees THROUGH the plastic) | 19016 | -18.04 | +0.32 |
| - propellers only | 15860 | -18.35 | +0.01 |
| metal core only (motor + battery + PCB + camera) | 6264 | -18.01 | +0.36 |
| dielectric only (no metal at all) | 22348 | -25.34 | -6.98 |

출처 ⟨outputs/report2_waveform_rcs.json : materials.rows⟩

![strip](outputs/figures/report2_materials.png)

**그림 8.** 드론에서 실제로 반사하는 것은 어느 부품인가?

In [ ]:
# 이 편의 숫자를 직접 열어보기 — 그림·표의 모든 값은 아래 JSON 에서 나온다.
import json
D = json.load(open('outputs/report02_derived.json'))
print(json.dumps(D['_meta']['definitions'], ensure_ascii=False, indent=1))
print('kr 최소 조합 :', D['electrical']['kr_min_drone'],
      D['electrical']['kr_min_band'], round(D['electrical']['kr_min'], 2))
print('밴드 기울기  :', {k: round(v, 3) for k, v in
      D['band_slope']['ours_db_per_ghz'].items()})
print('앵커 원장    :', {k: D['anchor'][k] for k in
      ('mode', 'correction_min_db', 'correction_max_db', 'largest_uncontrolled_db')})

## §6. 이 편의 한계

| 아직 안 되어 있는 것 | 다음 사람이 이어받을 지점 |
|---|---|
| PTD 모서리 회절항이 없다 — 밴드 기울기가 계통적으로 가파르다 | `src/rcs_sbr.py:184` 의 면적분에 등가 모서리 전류를 더한 뒤 §4 기울기를 다시 잰다 |
| 앵커의 크기전이 법칙(L² vs L⁴)이 미해소 — 최대 9.50 dB ⟨outputs/report02_derived.json : anchor.largest_uncontrolled_db⟩ 갈린다 | 06편 측정으로 두 기체(Matrice 4E · Mini 5 Pro)의 상대 레벨을 재 크기지수를 직접 고정한다 |
| 편파가 통제되지 않는다 — 커널은 재질당 스칼라 \|Γ\| 하나이고 앵커는 VV 측정이다 | 06편 측정 설계에서 편파 축을 확정한 뒤 `src/materials.py:171` 에 편파 분해를 넣을지 결정 |
| 앵커가 코드가 아니라 원장이다 — 생산 σ 는 여전히 SBR+PO 원값이다 | 05편 밴드 비교에서 `outputs/sigma_anchor.json` 의 `modes.slope_only.delta_db` 를 적용해 읽는다 |
| PO 모델 간극의 kr 문턱은 **구**에서만 잰 값이다(얇은 표적의 하한) | 평판·이면반사체 표준체로 같은 kr 스윕을 돌려 문턱을 다시 세운다 (`benchmark/verify_sbr_defect_fixes.py` 에 두 닫힌형이 이미 있다) |
| 메쉬 자세별 RMS 가 실물 대조와 5~10 dB 어긋난다 | 널을 인용하지 않는 현 규약을 유지하되 06편 측정으로 로브 위치부터 대조한다 |
| 모노스태틱만 방어된다 — 바이스태틱은 β≤45° 로 제한한다 | `src/rcs_sbr.py:330` `rcs_sbr_multistatic` 의 상반성 검사를 β 별로 다시 돌린다 |